# FMU Handling using FMPy

In [1]:
from fmpy import read_model_description, extract
import pint

In [2]:
# Define path and read model descriptions
ref_fmu_path = "Reference.fmu"
sensor_fmu_path = "AngleEncoder.fmu"

ref_md = read_model_description(ref_fmu_path)
sensor_md = read_model_description(sensor_fmu_path)

ref_vars_list = ref_md.modelVariables
sensor_vars_list= sensor_md.modelVariables

In [3]:
# Get the causality and variability of the variables
ref_var_dict = {var.name: var for var in ref_vars_list}
sensor_var_dict = {var.name: var for var in sensor_vars_list}

q_ref = ref_var_dict['q_ref']
q_sensor = sensor_var_dict['q']

# Check if q_ref is an input or output
print(f"q_ref causality: {q_ref.causality}, variability: {q_ref.variability}")
print(f"q_sensor causality: {q_sensor.causality}, variability: {q_sensor.variability}")

q_ref causality: output, variability: continuous
q_sensor causality: input, variability: continuous


In [4]:
from fmpy.model_description import ModelVariable

# Get possible types
print(f"q_ref type: {q_ref.type}, declaredType: {q_ref.declaredType}")
print(f"q_sensor type: {q_sensor.type}, declaredType: {q_sensor.declaredType}")

q_ref type: Real, declaredType: None
q_sensor type: Real, declaredType: None


In [5]:
q_ref.type

'Real'

In [6]:
q_ref.declaredType

In [7]:
sensor_var_dict['nBits'].type

'Real'

In [8]:
from pint import UnitRegistry
ureg = UnitRegistry()
ureg.formatter.default_format = '~P'
q_ref.unit = ureg.Unit(q_ref.unit)

print(type(q_ref.unit))

<class 'pint.Unit'>


In [9]:
# Replace the unit string 'rad' with the corresponding pint unit
ureg = pint.UnitRegistry()
ureg.formatter.default_format = '~P'  # Pretty print units

q_ref_unit = ureg.Unit(str(q_ref.unit))
q_sensor_unit = ureg.Unit(str(q_sensor.unit))

# Set the pint unit as unit attribute
q_ref.unit = q_ref_unit
q_sensor.unit = q_sensor_unit

print(q_ref_unit)
print(q_sensor_unit)

rad
rad


In [10]:
def connect(fmu1_var, fmu2_var):
    # Check if the causality and variability are compatible
    if fmu1_var.causality != 'output' or fmu2_var.causality != 'input':
        raise ValueError(f"Incompatible causality: {fmu1_var.causality} -> {fmu2_var.causality}")
    if fmu1_var.variability != fmu2_var.variability:
        raise ValueError(f"Incompatible variability: {fmu1_var.variability} != {fmu2_var.variability}")
    
    # Check if the units are compatible
    if fmu1_var.unit != fmu2_var.unit:
        raise ValueError(f"Units do not match: {fmu1_var.unit} != {fmu2_var.unit}")
    print(f"Connected {fmu1_var.name} to {fmu2_var.name} with unit {fmu1_var.unit}")

connect(q_ref, q_sensor)

Connected q_ref to q with unit rad


In [11]:
def extract_variables(fmu_path):
    var_dict = {var.name: var for var in read_model_description(fmu_path).modelVariables}

    for name, var in var_dict.items():
        unit = '' if var.unit is None else var.unit
        var.unit = ureg.Unit(str(unit))
    
    return var_dict

vars = extract_variables(ref_fmu_path)

for var_name, var in vars.items():
    print(f"{var_name}: causality={var.causality}, variability={var.variability}, unit={var.unit}")

der(_D_outputAlias_q_ref): causality=local, variability=continuous, unit=
q_ref: causality=output, variability=continuous, unit=rad
amplitude: causality=parameter, variability=fixed, unit=rad
frequency: causality=parameter, variability=fixed, unit=Hz
mean: causality=parameter, variability=fixed, unit=rad


In [12]:
from fmpy.fmi2 import FMU2Slave
import numpy as np

unzip_ref = extract(ref_fmu_path)
ref_fmu = FMU2Slave(guid=ref_md.guid,
                     unzipDirectory=unzip_ref,
                     modelIdentifier=ref_md.coSimulation.modelIdentifier)

ref_fmu.instantiate()
ref_fmu.setupExperiment()
ref_fmu.enterInitializationMode()

# Set parameters
amplitude = np.pi / 3
mean = 0.5
frequency = 0.33

q_ref_vr = ref_var_dict['q_ref'].valueReference
amplitude_vr = ref_var_dict['amplitude'].valueReference
mean_vr = ref_var_dict['mean'].valueReference
frequency_vr = ref_var_dict['frequency'].valueReference

ref_fmu.setReal([amplitude_vr], [amplitude])
ref_fmu.setReal([mean_vr], [mean])
ref_fmu.setReal([frequency_vr], [frequency])

print(f"New amplitude: {ref_fmu.getReal([amplitude_vr])[0]}")
print(f"New mean: {ref_fmu.getReal([mean_vr])[0]}")
print(f"New frequency: {ref_fmu.getReal([frequency_vr])[0]}")

ref_fmu.exitInitializationMode()

New amplitude: 1.0471975511965976
New mean: 0.5
New frequency: 0.33


0

In [13]:
q_ref_0 = ref_fmu.getReal([q_ref_vr])
ref_fmu.doStep(0, 0.1)
q_ref_1 = ref_fmu.getReal([q_ref_vr])
print(q_ref_0, q_ref_1)

[0.5] [0.715578819786763]


In [14]:
def initialize(path):
    md = read_model_description(path)
    unzipdir = extract(path)
    inst = FMU2Slave(guid=md.guid,
                            unzipDirectory=unzipdir,
                            modelIdentifier=md.coSimulation.modelIdentifier)
    inst.instantiate()

    vars = {var.name: var for var in md.modelVariables}

    # Transform unit strings to pint units
    for var in vars.values():
        unit = '' if var.unit is None else var.unit
        var.unit = ureg.Unit(str(unit)) # Convert unit to Pint instance

    inputs = {var.name: var for var in md.modelVariables if var.causality == 'input'}
    outputs = {var.name: var for var in md.modelVariables if var.causality == 'output'}
    parameters = {var.name: var for var in md.modelVariables if var.causality == 'parameter'}

    return inst, vars, inputs, outputs, parameters

inst, vars, inputs, outputs, parameters = initialize(ref_fmu_path)

type = vars['frequency'].type
unit = vars['frequency'].unit

value = 0.33 * ureg.hertz

compatible_unit = unit.is_compatible_with(value)

compatible_unit

True

In [15]:
from typing import List

def set(fmu, **signals):
    value_references: List[int] = []
    values: List[float] = []

    for key, value in signals.items():
        value_reference = vars[key].valueReference
        value_references.append(value_reference)
        values.append(value)

    fmu.setReal(value_references, values)

def get(fmu, *signals) -> List[float]:
    value_references: List[int] = []

    for key in signals:
        value_reference = vars[key].valueReference
        value_references.append(value_reference)

    return fmu.getReal(value_references)

set(ref_fmu, amplitude=np.pi/4, mean=0.0, frequency=0.5)

get(ref_fmu, 'amplitude', 'mean', 'frequency')

[0.7853981633974483, 0.0, 0.5]

**Model Structure and Dependencies**

In [16]:
fmu_path = 'Pendulum.fmu'
pend_md = read_model_description(fmu_path)

In [ ]:
init_unknowns = pend_md.initialUnknowns
for unknown in init_unknowns:
    unknown.dependencies
    print(f"Variable: {unknown.variable}, dependencies: {unknown.dependencies}")


Variable: ModelVariable(name='_D_outputAlias_omega_state', type='Real'), dependencies: [ModelVariable(name='omega0', type='Real')]
Variable: ModelVariable(name='_D_outputAlias_q_state', type='Real'), dependencies: [ModelVariable(name='q0', type='Real')]
Variable: ModelVariable(name='der(_D_outputAlias_omega_state)', type='Real'), dependencies: [ModelVariable(name='torque', type='Real'), ModelVariable(name='L', type='Real'), ModelVariable(name='m', type='Real'), ModelVariable(name='q0', type='Real')]
Variable: ModelVariable(name='der(_D_outputAlias_q_state)', type='Real'), dependencies: [ModelVariable(name='omega0', type='Real')]
Variable: ModelVariable(name='omega_state', type='Real'), dependencies: [ModelVariable(name='omega0', type='Real')]
Variable: ModelVariable(name='q_state', type='Real'), dependencies: [ModelVariable(name='q0', type='Real')]


In [19]:
# Get variable dependecies
pend_vars = {var.name: var for var in pend_md.modelVariables}